In [1]:
# Run surrogate predictions

from gpPredict import *
import pandas as pd
import numpy as np
import json

import matplotlib.pyplot as plt
import os

import pickle

from run_column_model import *

# Load pickle file with logistic regression
with open('log_reg_model.pkl', 'rb') as f:
    failure_mode_selection = pickle.load(f)
    # First parameter is the aspect ratio
    # Second parameter is Vp/Vs

    # Codes:
    # 0 = Flexure
    # 1 = Shear

# Use latex for plots
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

In [2]:
# ndParams = ['ar', 'lrr', 'srr', 'alr', 'sdr', 'smr']
# ndParams = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]

def get_BW_params(ndParams):

    # Define the failure mode using the logistic regression model
    failure_mode = failure_mode_selection.predict(np.array([[ndParams[0], ndParams[5]]]))[0]

    # Define the surrogate_file and input_json based on the failure mode
    if failure_mode == 0:
        print('Flexure failure mode')
        surrogate_file = os.path.join('gpModelFlexure', 'SimGpModel.json')
        input_json = os.path.join('gpModelFlexure', 'scInput.json')
    else:
        print('Shear failure mode')
        surrogate_file = os.path.join('gpModelShear', 'SimGpModel.json')
        input_json = os.path.join('gpModelShear', 'scInput.json')

    params_list = [
        ["RV_column1", ndParams[0]],
        ["RV_column2", ndParams[1]],
        ["RV_column3", ndParams[2]],
        ["RV_column4", ndParams[3]],
        ["RV_column5", ndParams[4]],
        ["RV_column6", ndParams[5]]
    ]

    output = main(params_list, [], surrogate_file, 'dummyout.out', input_json)
    
    # bw model parameters are the all the output parameters except the last one
    bw_model_params = (output[0][:-1].T).tolist()

    # min error is the last output parameter
    min_error = float(output[0][-1])
    
    return bw_model_params, min_error


In [3]:
# Open calibration_info.csv
calibration_info = pd.read_csv('calibration_info.csv')

# Open data_all.csv file
data_all = pd.read_csv('data_all.csv')

# Iterate over all rows in calibration_info
for ii in range(len(calibration_info)):
    # Get the UniqueId
    UniqueId = calibration_info['UniqueId'].iloc[ii]

    # Find UniqueId in data_all
    row = data_all[data_all['UniqueId'] == UniqueId]

    # For this row, get the calibrated bw model parameters
    calParams = calibration_info[['gamma', 'kappa', 'eta1', 'sig', 'lam', 'mup', 'sigp', 'rsmax', 'alpha', 'alpha1', 'alpha2', 'betam1', 'n', 'kappa_k']].iloc[ii].values

    # Now, from data_all, get the nondimensional parameters
    ndParams = row[['ar', 'lrr', 'srr', 'alr', 'sdr', 'smr']].values[0]

    # Get surrogate-predicted bw model parameters
    predParams, predErr = get_BW_params(ndParams)

    # Get the test_XXX.json file from test_data folder where XXX is the UniqueId
    with open('test_data/test_' + str(UniqueId).zfill(3) + '.json') as f:
        test_data = json.load(f)

    # Run the column model with the calParams
    calResults = run_model(test_data, calParams, do_plots=False)

    # Run the column model with the predParams
    surrResults = run_model(test_data, predParams, do_plots=False)

    print('Calibration Error: ', calResults['mae'])
    plt.figure()
    # Plot experimental data
    plt.plot(calResults['exp_data']['drift'], calResults['exp_data']['nforce'], label='Experimental', color='black', linewidth=1.0)
    # Plot from calResults drift vs normalized force
    plt.plot(calResults['sim_data']['drift'], calResults['sim_data']['nforce'], label='Calibration', color='red', linewidth=0.5, linestyle='--')
    # Plot from surrResults displacement vs normalized force
    plt.plot(surrResults['sim_data']['drift'], surrResults['sim_data']['nforce'], label='Surrogate', color='blue', linewidth=0.5, linestyle='--')
    plt.xlabel('Drift Ratio')
    plt.ylabel('Normalized Force')
    plt.legend()
    plt.title('UniqueId: ' + str(UniqueId) + ', Calibration Error: ' + format(calResults['mae'], '.4f') + ', Surrogate Error: ' + format(predErr, '.4f'))
    plt.show()    


Flexure failure mode
Finished... Run Time =  2.1123268604278564 sec
Finished... Run Time =  2.11575984954834 sec
Calibration Error:  0.04899548220494011


FileNotFoundError: Matplotlib's TeX implementation searched for a file named 'cmr10.tfm' in your texmf tree, but could not find it

<Figure size 640x480 with 1 Axes>

Flexure failure mode


KeyboardInterrupt: 

In [ ]:
predParams

([0.5393784154234877,
  1.0199400177461684,
  1.104441241540115,
  0.616409639544926,
  0.37116549031771146,
  1.4553139840157703,
  0.7603484138016328,
  0.7115309766700496,
  0.007785351405372367,
  4.752529423060971,
  0.6521372098889564,
  0.004759314839760715,
  3.6768105624456933,
  0.7496963863203646],
 0.0485990883398565)